# Inference time for yolo

In [1]:
!python ./yolov5/val.py --img 512 --data ./yolov5/data/custom_512.yaml --weights ./yolov5/runs/train/exp9/weights/best.pt

val: data=./yolov5/data/custom_512.yaml, weights=['./yolov5/runs/train/exp9/weights/best.pt'], batch_size=32, imgsz=512, conf_thres=0.001, iou_thres=0.6, task=val, device=, workers=8, single_cls=False, augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, save_json=False, project=yolov5\runs\val, name=exp, exist_ok=False, half=False, dnn=False
'git' is not recognized as an internal or external command,
operable program or batch file.
YOLOv5  2022-4-15 torch 1.10.0+cu113 CUDA:0 (NVIDIA GeForce GTX 1060, 6144MiB)

Fusing layers... 
Model summary: 213 layers, 7012822 parameters, 0 gradients, 15.8 GFLOPs

val: Scanning 'C:\Users\morti\Documents\School\master\Masterproef\source\yolov5\customdata_512\labels\val.cache' images and labels... 133 found, 267 missing, 0 empty, 0 corrupt: 100%|██████████| 400/400 [00:00<?, ?it/s]
val: Scanning 'C:\Users\morti\Documents\School\master\Masterproef\source\yolov5\customdata_512\labels\val.cache' images and labels... 133 found

Benchmark YOLOV5 for 10000 images (512x512)

In [ ]:
import torch
import time
import pandas as pd
import skimage.io
import glob
import os
import re
numbers = re.compile(r'(\d+)')
def numericalSort(value):
    parts = numbers.split(value)
    parts[1::2] = map(int, parts[1::2])
    return parts


batchsize = 150
img_count = 12000 #veelvoud van batchsize
folder = "./data/data_512/images_512/"
image_list = []
teller = 0
for filename in sorted(os.listdir(folder), key=numericalSort):
    im=skimage.io.imread(os.path.join(folder,filename))
    image_list.append(im)
    teller += 1
    print(filename)
    if teller % batchsize == 0:
        print("----------------------------------------")
    if teller == img_count:
        break

In [2]:
model = torch.hub.load('ultralytics/yolov5', 'custom', path='./yolov5/runs/train/size512_epoch50_batch16/weights/best.pt')
device = torch.device("cuda")
model.to(device)
times = []
_times = []
factor = img_count//batchsize
#opwarming
_ = model(image_list[0:batchsize])
#meting
for i in range(10):
    for i in range(1,factor+1):
        starter, ender = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
        with torch.no_grad():
            starter.record()
            start = time.time()
            _ = model(image_list[batchsize*(i-1):batchsize*i])
            ender.record()
            torch.cuda.synchronize()
            times.append(starter.elapsed_time(ender))
            _times.append(time.time() - start)

df = pd.DataFrame({"inference time per HR image (ms)": times, "with default time (s)": _times})

df.to_csv("performance_per_image_10_epochs.csv")

Using cache found in C:\Users\morti/.cache\torch\hub\ultralytics_yolov5_master
YOLOv5  2021-12-30 torch 1.10.0+cu113 CUDA:0 (NVIDIA GeForce GTX 1060, 6144MiB)

Fusing layers... 
Model Summary: 213 layers, 7012822 parameters, 0 gradients, 15.8 GFLOPs
Adding AutoShape... 
